# Lab 3: Calidad de datos y feature engineering distribuido — respuestas revisadas
**Análisis de Big Data · Magíster en Data Science UDD · Sesión 3 (vie 21 ago)**

**Nombre:** Dante Aguirre Bravo.

**Objetivos:** convertir las seis dimensiones de calidad en **reglas ejecutables** que devuelven un conteo, producir el **reporte de calidad con columna de decisión** y limpiar con criterio (filtrar, imputar con flag, deduplicar por ventana).

**Entorno:** Google Colab (ejecutar el setup) o Databricks CE (saltar el <code>pip install</code> de pyspark).

**Dataset:** NYC Yellow Taxi entre **enero y marzo 2023** (~9,5M viajes, 3 archivos Parquet, ~170 MB). Tres meses, no doce, para que el laboratorio corra en el bloque de clase.

**Entrega:** notebook ejecutado completamente (Kernel → Restart & Run All antes de subir). **Cuenta para el 25% de laboratorios.**

**Rúbrica:** corre completamente ✓ · reporte de calidad con **decisión justificada** por regla ✓ · ninguna regla sobre el umbral filtrada sin justificación escrita (sección 3.4) ✓ · shuffle identificado en el <code>EXPLAIN</code> ✓ · uso de IA declarado ✓

> **Alcance de esta versión:** revisión de las respuestas en texto a partir de la evidencia guardada. Código y resultados conservados íntegramente; no se realizó una nueva ejecución.

> **v2 · Actualizaciones tras la sesión 3.** (a) Nueva sección **0c**: qué ocurre realmente cuando se castea un texto a `double`, pregunta que quedó abierta en clase. (b) Sección **3** reescrita: el filtrado ahora **respeta el umbral visto en clases**.

> Continuidad con el Lab 2: mismo dataset y mismo motor. Allí la pregunta era *cuál motor conviene*, ahora es *si los datos que ese motor lee son confiables y cómo se convierten en variables*.


## 0. Setup

In [ ]:
# Colab / Jupyter local (en Databricks CE, no reinstalar pyspark)
%pip install -q pyspark==3.5.4 pandas


In [ ]:
import urllib.request, os, glob
MESES = 3                      # ene-mar 2023. Subirlo exige ajustar PERIODO_FIN más abajo.
PERIODO_INI, PERIODO_FIN = "2023-01-01", "2023-04-01"
CORTE_TEST = "2023-03-01"      # train: ene-feb · test: mar

for m in range(1, MESES + 1):
    f = f"yellow_tripdata_2023-{m:02d}.parquet"
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"https://d37ci6vzurychx.cloudfront.net/trip-data/{f}", f)
        print("ok", f)
ARCHIVOS = sorted(glob.glob("yellow_tripdata_2023-*.parquet"))
print(len(ARCHIVOS), "archivos ·", round(sum(os.path.getsize(a) for a in ARCHIVOS)/1e6), "MB")


In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
from functools import reduce
import pandas as pd, time

spark = (SparkSession.builder
         .master("local[2]")
         .appName("ADBD-Lab3")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "64")
         .getOrCreate())
pd.set_option("display.width", 160)

### 0b. Carga con esquema declarado: la primera regla de calidad

Hallazgo del Lab 2: los Parquet de TLC **cambian tipos y nombres de columna entre meses**. Leer los archivos juntos y confiar en la inferencia es la primera falla de calidad, y ocurre antes de mirar un solo dato.

El patrón correcto es el de abajo: **esquema esperado declarado en el código**, cada archivo leído con el suyo, cast explícito. Esto es la dimensión **Esquema** que se verifica en la sección 2.

In [ ]:
# Esquema esperado: nombre -> tipo. Es un contrato, no una sugerencia.
COLS = {
    "VendorID": "bigint",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "double",
    "trip_distance": "double",
    "RatecodeID": "double",
    "PULocationID": "bigint",
    "DOLocationID": "bigint",
    "payment_type": "bigint",
    "fare_amount": "double",
    "tip_amount": "double",
    "total_amount": "double",
}

partes = [spark.read.parquet(a).select([F.col(c).cast(t).alias(c) for c, t in COLS.items()])
          for a in ARCHIVOS]
bronze = reduce(lambda a, b: a.unionByName(b), partes).cache()   # capa Bronze: crudo, sin juicio
n_bronze = bronze.count()
print(f"Bronze: {n_bronze:,} filas · {len(bronze.columns)} columnas")


### 0c. ¿Y si el cast no puede? Pregunta abierta en la sesión 3

> **Si se declara `double` y la columna venía como texto, ¿el cast falla?**

**No falla: devuelve `NULL`, en silencio,** que es el peor comportamiento posible, pues suele detectarse cuando el modelo ya está entrenando con muchos datos nulos.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

crudo = spark.createDataFrame(
    [("12.5",), ("0",), ("3,7",), ("N/A",), ("",), (None,), ("1e3",), (" 8.0 ",), ("$15",), ("9999999999999999999",)],
    StructType([StructField("monto_txt", StringType())]))

(crudo.withColumn("a_double",   F.col("monto_txt").cast("double"))
      .withColumn("try_cast",   F.expr("try_cast(monto_txt AS double)"))
      .withColumn("a_int",      F.col("monto_txt").cast("int"))
      .show(truncate=False))


**Fila por fila, lo que acaba de pasar:**

| Entrada | `cast("double")` | Por qué importa |
|---|---|---|
| `"12.5"`, `"1e3"`, `" 8.0 "` | 12.5 · 1000.0 · 8.0 | Parsea notación científica y tolera espacios. |
| `"3,7"` | **null** | **Coma decimal.** Un CSV exportado con configuración regional chilena pierde *toda* la columna, sin avisar el error. |
| `"N/A"`, `""`, `"$15"` | **null** | El centinela textual, el vacío y el símbolo de moneda son indistinguibles de un dato ausente después del cast. |
| `"9999999999999999999"` | 1.0E19 en `double`, **null** en `int` | El desborde tampoco avisa: cambia según el tipo destino. |

Tres consecuencias prácticas:

1. **El nulo de la sección 1 puede no venir del origen.** Puede haberlo fabricado la línea de carga. Son dos problemas distintos y se arreglan distinto.
2. `try_cast` devuelve `NULL` **por defecto** al no poder hacer el *casting*, por lo que quien lea el código sabe que la generación de **null** es un comportamento deseado. `cast` a secas, en cambio, no distingue entre una decisión y un descuido.
3. La conversión de `double` a entero **trunca, no redondea** (`8.9 → 8`), y el desborde vuelve a dar `NULL`.

In [ ]:
# Modo ANSI: el MISMO cast lanza excepción en vez de devolver null.
# Opcional hasta Spark 3.x, pero pasa a ser el DEFAULT en Spark 4.0. Por lo
# tanto, el mismo notebook cambia de comportamiento al migrar de versión, sin
# tocar una línea.
spark.conf.set("spark.sql.ansi.enabled", True)
try:
    crudo.select(F.col("monto_txt").cast("double")).collect()
    print("sin error")
except Exception as e:
    print("ANSI lanza ->", type(e).__name__, "|", str(e).split("\n")[0][:150])
finally:
    spark.conf.set("spark.sql.ansi.enabled", False)


Con ANSI activado, el comportamiento pasa a ser el descrito en clases. Existen dos opciones: **fallar de forma temprana y ruidosa**, o **seguir y perder datos**. Es responsabilidad nuestra saber en cuál de los dos modos corre el pipeline.

In [ ]:
# La regla operativa: un cast es una transformación y, como toda transformación, se audita.
# Nulos ANTES vs DESPUÉS: la diferencia son filas que destruyó la línea de carga.
def auditar_cast(path, cols):
    raw = spark.read.parquet(path)
    presentes = [c for c in cols if c in raw.columns]
    nulos = lambda df: df.select([F.sum(F.col(c).isNull().cast("long")).alias(c) for c in presentes]).collect()[0].asDict()
    antes = nulos(raw)
    despues = nulos(raw.select([F.col(c).cast(cols[c]).alias(c) for c in presentes]))
    return pd.DataFrame([(c, raw.schema[c].dataType.simpleString(), cols[c],
                          antes[c], despues[c], despues[c] - antes[c]) for c in presentes],
                        columns=["columna", "tipo_origen", "tipo_declarado",
                                 "nulos_antes", "nulos_despues", "creados_por_el_cast"])

audit = auditar_cast(ARCHIVOS[0], COLS)
NULOS_POR_CAST = int(audit["creados_por_el_cast"].sum())
print("Nulos fabricados por el cast:", NULOS_POR_CAST)
audit


**Leer la columna `creados_por_el_cast`:** en este caso es cero, ya que los tipos de las columnas coinciden y el *casting* no destruye nada. Ahora no existe un supuesto, sino **la certeza** de que ocurre lo esperado.

`NULOS_POR_CAST` entra al reporte de calidad de la sección 2 como una regla más, bajo la dimensión **Esquema**. Es la única regla que mide los datos anulados (o "el daño causado") por el propio pipeline.

## 1. Etapa 1: Perfilado en una sola pasada

Nulos, cardinalidades y rangos. La restricción que define el ejercicio: **una sola pasada sobre los datos**. Un <code>count()</code> por columna sobre 9,5M filas son doce recorridos completos del dataset. El mismo resultado se obtiene con una sola agregación de doce expresiones.

In [ ]:
# Nulos y cardinalidad aproximada: dos pasadas, no veinticuatro.
nulos = bronze.select([F.sum(F.col(c).isNull().cast("long")).alias(c) for c in bronze.columns]).collect()[0].asDict()
card  = bronze.select([F.approx_count_distinct(c).alias(c) for c in bronze.columns]).collect()[0].asDict()

perfil = pd.DataFrame({"columna": list(COLS), "tipo": list(COLS.values())})
perfil["nulos"] = perfil["columna"].map(nulos)
perfil["%_nulos"] = (perfil["nulos"] / n_bronze * 100).round(2)
perfil["card_aprox"] = perfil["columna"].map(card)
perfil


In [ ]:
# Rangos: el mínimo y el máximo son donde vive la basura.
NUMERICAS = [c for c, t in COLS.items() if t == "double" or c.endswith("ID")]
bronze.select(NUMERICAS).summary("min", "25%", "50%", "75%", "max").toPandas()


**Lectura del perfilado**

**1. ¿Qué columnas tienen nulos y alguna supera el 5%?**

Sobre **9.384.487 registros**, `passenger_count` y `RatecodeID` tienen **236.179 nulos cada una (2,52%)**. Las otras diez columnas perfiladas no presentan nulos y ninguna supera el 5%. Tener el mismo conteo no demuestra, por sí solo, que los valores falten en las mismas filas. Además, completitud no equivale a calidad total: una columna sin nulos todavía puede contener valores inválidos.

**2. ¿Qué significan los mínimos de monto y distancia?**

El mínimo de `total_amount` es **−982,95 dólares**. Puede corresponder a una reversa, devolución, ajuste o error de registro; el perfil no permite confirmar cuál de esas causas lo explica. Para el análisis de viajes con cobro positivo, el laboratorio lo considera fuera del rango válido. Esta decisión delimita el universo analítico, pero no demuestra que la transacción sea ficticia.

El mínimo de `trip_distance` es **0 millas**: no hay distancia registrada. Puede tratarse de un viaje cancelado, un cobro sin desplazamiento o una falla de captura. Un monto positivo tampoco prueba que se haya realizado un viaje; por eso se revisa el segmento junto con su duración y velocidad antes de decidir.

**3. ¿El máximo de distancia es un viaje o un error? ¿Con qué criterio se decide?**

Las **335.004,33 millas** observadas son incompatibles con un viaje ordinario de taxi y constituyen una anomalía extrema de registro o medición. El criterio reproducible combina el rango declarado de **0,01 a 200 millas** con duración, orden temporal y velocidad. El límite de 200 millas es una regla adoptada por este laboratorio, no una frontera física universal. También debe distinguirse el caso extremo del conjunto completo de filas que incumple la regla: no todas tienen necesariamente la misma causa.

**4. ¿La cardinalidad de `PULocationID` coincide con las aproximadamente 265 zonas esperadas?**

La salida muestra una **cardinalidad aproximada de 264**, compatible en orden de magnitud con la referencia de la consigna. Se calculó con `approx_count_distinct`; por eso no permite afirmar que falte exactamente una zona. El rango de identificadores llega a 265, pero el máximo tampoco informa cuántos códigos diferentes existen. Para verificar la cobertura exacta se necesitarían un conteo distinto exacto y la comparación con el catálogo de zonas.

## 2. Etapa 2: Diccionario de reglas y reporte de calidad

Dos columnas de apoyo primero (<code>duracion_min</code> y <code>velocidad_mph</code>): sin ellas, las reglas de consistencia no se pueden escribir. Son parte del **diagnóstico**, no del set de features.

In [ ]:
diag = (bronze
    .withColumn("duracion_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0)
    .withColumn("velocidad_mph",
        F.when(F.col("duracion_min") > 0, F.col("trip_distance") / (F.col("duracion_min") / 60.0))))


In [ ]:
# Diccionario de reglas: nombre -> (dimensión, condición que debe cumplirse, decisión propuesta)
REGLAS = {
    "completitud_pasajeros":      ("Completitud",  F.col("passenger_count").isNotNull(),                        "imputar + flag"),
    "completitud_zona_origen":    ("Completitud",  F.col("PULocationID").isNotNull(),                           "eliminar"),
    "validez_monto":              ("Validez",      F.col("total_amount").between(0.01, 5000),                   "eliminar"),
    "validez_distancia":          ("Validez",      F.col("trip_distance").between(0.01, 200),                   "eliminar"),
    "validez_medio_pago":         ("Validez",      F.col("payment_type").isin([1, 2, 3, 4, 5, 6]),              "eliminar"),
    "consistencia_orden_tiempos": ("Consistencia", F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"), "eliminar"),
    "consistencia_duracion":      ("Consistencia", F.col("duracion_min").between(1, 360),                       "eliminar"),
    "plausibilidad_velocidad":    ("Consistencia", F.col("velocidad_mph") <= 100,                               "eliminar"),
    "actualidad_periodo":         ("Actualidad",   (F.col("tpep_pickup_datetime") >= PERIODO_INI) &
                                                   (F.col("tpep_pickup_datetime") <  PERIODO_FIN),              "eliminar"),
}

# Una sola pasada para las nueve reglas.
# Ojo con los nulos: 'when(cond, 0).otherwise(1)' cuenta el nulo como FALLA (una condición
# nula no es una condición cumplida). Con sum(~cond) el nulo se ignoraría y el reporte mentiría.
exprs = [F.sum(F.when(cond, 0).otherwise(1)).alias(nombre) for nombre, (_, cond, _) in REGLAS.items()]
fallos = diag.agg(*exprs).collect()[0].asDict()


In [ ]:
# Unicidad: se mide aparte porque cuesta un shuffle (countDistinct sobre la clave de negocio).
CLAVE = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"]
n_distintas = diag.select(CLAVE).distinct().count()
fallos["unicidad_clave_viaje"] = n_bronze - n_distintas

# Esquema: contrato declarado vs. observado.
observado = {f.name: f.dataType.simpleString() for f in bronze.schema.fields}
desvios = {c: (t, observado.get(c)) for c, t in COLS.items() if observado.get(c) != t}
print("Desvíos de esquema:", desvios if desvios else "ninguno (el cast de la carga los absorbió)")


In [ ]:
filas = [(dim, nombre, fallos[nombre], round(fallos[nombre] / n_bronze * 100, 2), dec)
         for nombre, (dim, _, dec) in REGLAS.items()]
filas.append(("Unicidad", "unicidad_clave_viaje", fallos["unicidad_clave_viaje"],
              round(fallos["unicidad_clave_viaje"] / n_bronze * 100, 2), "deduplicar por ventana"))
# Sección 0c: nulos que fabricó el pipeline de carga, no el origen.
filas.append(("Esquema", "cast_sin_perdida", NULOS_POR_CAST,
              round(NULOS_POR_CAST / n_bronze * 100, 2),
              "revisar la declaración de tipos" if NULOS_POR_CAST else "sin acción"))

reporte = pd.DataFrame(filas, columns=["dimensión", "regla", "filas_fallidas", "%", "decisión"])
reporte = reporte.sort_values("%", ascending=False).reset_index(drop=True)
reporte


### 2b. Interpretación del reporte y decisiones justificadas

**1. ¿Qué regla tiene el mayor porcentaje de fallas y cuál es la causa?**

`completitud_pasajeros` y `validez_medio_pago` empatan con **236.179 fallas (2,52% del Bronze)**. La primera detecta pasajeros ausentes; la segunda, un medio de pago que no pertenece al conjunto aceptado por el código. No son necesariamente el mismo tipo de problema.

El diagnóstico posterior aporta una relación más concreta: entre los **234.511 casos** de medio de pago inválido que sobreviven al filtrado automático, `passenger_count` no tiene ningún valor informado. Sus medianas son **2,61 millas, 16 minutos y 23,62 dólares**. Esto sugiere un patrón sistemático de captura y viajes potencialmente aprovechables; no demuestra qué sistema originó el patrón. Identificar la causa requeriría revisar el código de pago, el proveedor y el mes de origen.

**2. ¿Cambian las decisiones propuestas al observar los porcentajes?**

Sí. El reporte impreso presenta la propuesta inicial de eliminar, mientras que `DECISIONES` conserva con flag tres grupos. El siguiente cuadro expresa la decisión efectivamente aplicada:

| Regla | Fallas sobre Bronze | Decisión final | Justificación |
|---|---:|---|---|
| Completitud de pasajeros | 236.179 (2,52%) | Imputar + flag | Evita eliminar registros por la ausencia de una variable; la mediana se aprende solo con train. |
| Validez del medio de pago | 236.179 (2,52%) | Conservar con flag | El grupo presenta distancias, duraciones y montos plausibles. La selección de tarjeta se hace posteriormente al definir el universo del objetivo. |
| Validez de distancia | 135.731 (1,45%) | Conservar con flag | La regla mezcla distancia cero y valores extremos; un único filtro borraría casos heterogéneos sin resolver su causa. |
| Consistencia de duración | 112.340 (1,20%) | Conservar con flag | Duraciones muy cortas y muy largas requieren interpretación; el monto positivo no basta para clasificarlas como viajes válidos. |
| Validez de monto | 81.601 (0,87%) | Filtrar | Delimita el análisis a cobros dentro del rango declarado de 0,01 a 5.000 dólares. |
| Plausibilidad de velocidad | 9.033 (0,10%) | Filtrar | Aplica el umbral declarado de hasta 100 mph y trata una condición nula como falla. |
| Orden de tiempos | 3.839 (0,04%) | Filtrar | Exige que el término sea posterior al inicio para calcular una duración positiva. |
| Completitud de zona de origen | 0 (0,00%) | Filtrar si incumple | La regla está activa, pero no elimina filas por esta causa en la ejecución guardada. |
| Actualidad del período | 139 (0,00% redondeado) | Filtrar | Excluye inicios fuera de enero–marzo de 2023. El porcentaje redondeado no significa cero casos. |
| Unicidad de la clave | 70.121 (0,75%) | Deduplicar por ventana | El reporte mide repeticiones en Bronze; después del filtrado quedan 18 excedentes por clave en Silver. |
| Cast sin pérdida | 0 | Sin acción en la auditoría realizada | No se crearon nulos en el archivo auditado; el código revisa `ARCHIVOS[0]`, no los tres meses. |

Se aplica la política del código: **hasta 1%**, filtrado automático de las reglas elegibles; **sobre 1%**, decisión explícita; **sobre 5%**, entender la causa antes de descartar. Ninguna regla supera el 5% en esta salida. La frecuencia orienta la revisión, pero no demuestra la causa del problema.

Los conteos por regla **no deben sumarse como pérdidas**, porque una fila puede incumplir varias reglas. El efecto conjunto observado del filtrado automático es pasar de **9.384.487 a 9.294.106 filas**: se excluyen **90.381 (0,96%)**.

**3. ¿Qué se perdería al eliminar las filas sin pasajeros?**

Se perderían **236.179 registros del Bronze**, incluidos casos con información utilizable de distancia, duración y monto. Si la ausencia está asociada a una forma particular de registro, la eliminación puede alterar la composición de los datos. Imputar conserva la observación y el flag distingue un dato observado de uno completado. La imputación no recupera el número verdadero de pasajeros: introduce una aproximación aprendida exclusivamente con train.

Hay una distinción relevante en este notebook: conservar el segmento en Silver no implica que todo llegue al modelo. Gold selecciona `payment_type = 1`, por lo que los medios de pago fuera del catálogo quedan fuera de ese universo por una decisión posterior.

**4. ¿Qué regla adicional necesita el dataset?**

Propondría una regla de **validez del número de pasajeros** que separe un dato ausente de uno presente pero incompatible con el contrato del servicio. El perfil tiene mínimo **0** y máximo **9**, mientras la regla actual solo verifica que el valor no sea nulo. La nueva regla exigiría un valor entero positivo dentro de un límite documentado para el servicio analizado. Antes de fijar ese límite se debe confirmar la capacidad admitida; el perfil por sí solo no permite declarar inválido todo valor 9. Se trata de una propuesta analítica y no de una regla implementada en el código conservado.

## 3. Etapa 3: Limpieza: filtrar, marcar, deduplicar

Aquí se ejecuta la columna `decisión` de la sección 2b. Y la primera decisión es **cuáles reglas tienen permiso para descartar filas**: el criterio de la clase (≤1% se filtra sin discusión, >5% no se toca hasta entender) tiene que estar **en el código**, no solo en la lámina. Un pipeline que descarta el 8% de las filas sin que nadie lo note es exactamente lo que la sección 2 intenta prevenir.

In [ ]:
# 3.1 La política de la sección 2b, ejecutable:
#   <= 1%   -> se filtra sin discusión
#   1% a 5% -> se filtra SOLO con justificación escrita
#   > 5%    -> no se filtra hasta entender la causa
UMBRAL_LIBRE, UMBRAL_ALERTA = 1.0, 5.0
pct = {r: fallos[r] / n_bronze * 100 for r in REGLAS}

# 'completitud_pasajeros' queda fuera: su decisión fue imputar, no descartar.
REGLAS_FILTRO = {n: v for n, v in REGLAS.items() if n != "completitud_pasajeros"}
automaticas  = [r for r in REGLAS_FILTRO if pct[r] <= UMBRAL_LIBRE]
a_justificar = [r for r in REGLAS_FILTRO if pct[r] >  UMBRAL_LIBRE]

print(f"Filtrado automático — {len(automaticas)} reglas bajo el {UMBRAL_LIBRE}%:")
for r in automaticas:
    print(f"    {r:<30}{pct[r]:>7.2f}%")
print(f"\nRequieren decisión explícita — {len(a_justificar)} reglas:")
for r in a_justificar:
    alerta = "  <-- SOBRE EL 5%: entender antes de tocar" if pct[r] > UMBRAL_ALERTA else ""
    print(f"    {r:<30}{pct[r]:>7.2f}%{alerta}")


In [ ]:
# 3.2 Se filtra SOLO con las reglas que están bajo el umbral. Las demás siguen adentro.
cond_auto = (reduce(lambda a, b: a & b, [F.coalesce(REGLAS[r][1], F.lit(False)) for r in automaticas])
             if automaticas else F.lit(True))
silver = diag.filter(cond_auto)
n_auto = silver.count()
print(f"Bronze {n_bronze:,} -> tras filtrado automático {n_auto:,} ({(1 - n_auto/n_bronze)*100:.2f}% descartado)")


In [ ]:
# 3.3 Evidencia antes de decidir: qué hay dentro de lo que cada regla pendiente quiere descartar.
for r in a_justificar:
    print(f"\n=== {r} · {pct[r]:.2f}% de las filas ===")
    (silver.filter(~F.coalesce(REGLAS[r][1], F.lit(False)))
           .select("trip_distance", "duracion_min", "velocidad_mph", "total_amount", "passenger_count")
           .summary("count", "min", "50%", "max").show())


**3.3. ¿Son datos completamente rotos o existe un posible segmento legítimo?**

La evidencia muestra grupos heterogéneos. Las estadísticas siguientes corresponden a las filas que **ya sobrevivieron al filtrado automático**, por lo que sus tamaños son menores que los del reporte inicial.

| Regla incumplida | Casos en el diagnóstico | Mediana de distancia | Mediana de duración | Mediana de monto |
|---|---:|---:|---:|---:|
| Validez de distancia | 121.596 | 0 millas | 1,8 min | 20,02 dólares |
| Validez del medio de pago | 234.511 | 2,61 millas | 16 min | 23,62 dólares |
| Consistencia de duración | 88.106 | 0 millas | 0,25 min | 15,40 dólares |

El segmento con medio de pago inválido presenta medianas compatibles con viajes ordinarios y ausencia de pasajeros en todos sus registros. Es razonable investigar una diferencia de captura antes de eliminarlo.

En los otros dos grupos, distancia cero, duración corta y monto positivo son compatibles con varias explicaciones: cancelaciones, cobros sin desplazamiento o errores de registro. Son hipótesis, no causas verificadas. Además, el diagnóstico muestra extremos como **10.029,18 minutos** de duración que requieren revisión específica.

La decisión de conservar con flag evita declarar inválido un grupo completo con evidencia insuficiente. **Conservar no equivale a certificar calidad**: permite mantener trazabilidad y distinguir estos casos en un análisis posterior. Tampoco debe suponerse que cada fila reúne simultáneamente las medianas de la tabla; son resúmenes calculados por columna.

### 3.3b Decisión por regla

Cada regla pendiente necesita **una acción y una justificación escrita**. Las dos acciones posibles:

- **`filtrar`**: las filas se descartan. Exige justificación: qué son esas filas y por qué no pertenecen al análisis.
- **`conservar_con_flag`**: las filas se quedan con una columna `incumple_<regla>` que las marca. Es la opción correcta cuando lo que la regla llama "error" resulta ser un segmento real del negocio: el modelo puede usar la marca como señal.

Editar el diccionario de la celda siguiente. Mientras una justificación diga `PENDIENTE`, el notebook lo reporta y **la entrega está incompleta**.

## 3.4. Justificación de las decisiones adoptadas

Las tres reglas sobre el umbral automático se conservan con flag, de acuerdo con el diccionario de la celda siguiente:

- **Distancia:** se mantiene el registro porque el incumplimiento reúne distancia cero y extremos con posibles causas diferentes. La marca permite identificar el grupo sin afirmar que todos sean viajes válidos. En el cálculo posterior de tarifa por milla, una distancia cero sigue requiriendo atención; conservar una fila no resuelve todas sus variables derivadas.
- **Medio de pago:** se mantiene en Silver porque el diagnóstico contiene transacciones plausibles. El flag preserva la anomalía y la posterior selección de tarjeta define la población donde se analiza la propina; esa selección no es una corrección del código de pago.
- **Duración:** se conserva el conjunto para evitar atribuir una misma causa a registros muy cortos y muy largos. La marca permite distinguirlo y deja abierta una revisión específica de los extremos.

La salida confirma la creación de las tres columnas `incumple_...` y mantiene **9.294.106 filas** antes de deduplicar. Estas columnas registran la decisión de calidad, pero el ensamblador de features conservado no las incorpora automáticamente al vector del modelo. El flag de pasajeros sí está incluido en `NUM_DIRECTAS`.

In [ ]:
DECISIONES = {
    "validez_distancia": (
        "conservar_con_flag",
        "La regla falla en 1,45%. La mediana del segmento es distancia 0, duración 1,8 min y "
        "monto $20,02: mezcla posibles cancelaciones, cobros mínimos o tarifas fijas con anomalías. "
        "Se conserva para no borrar transacciones reales y se marca para análisis/modelado."
    ),
    "validez_medio_pago": (
        "conservar_con_flag",
        "La regla falla en 2,52%. Sus medianas (2,61 millas, 16 min y $23,62) son plausibles y "
        "passenger_count falta en todo el segmento, señal de un proceso de captura distinto. "
        "Se conserva con flag; el modelo de propina luego selecciona explícitamente pago con tarjeta."
    ),
    "consistencia_duracion": (
        "conservar_con_flag",
        "La regla falla en 1,20%. Los casos cortos con monto positivo pueden ser cancelaciones o cobros "
        "mínimos legítimos, mientras los extremos largos pueden ser taxímetros mal cerrados. El flag "
        "permite separar y estudiar ambos casos sin eliminar todo el segmento."
    ),
}

for r in a_justificar:
    assert r in DECISIONES, f"falta la decisión de {r}"
for r, (accion, just) in DECISIONES.items():
    assert accion in ("filtrar", "conservar_con_flag"), f"acción inválida en {r}: {accion}"
    if just.startswith("PENDIENTE"):
        print(f"[!] {r} ({pct[r]:.2f}%) sin justificación — la entrega no está completa")


## 3.5 Aplicación de las decisiones de calidad de datos.
Esta sección ejecuta las acciones definidas en el diccionario `DECISIONES` para las reglas que requerían una justificación explícita (`a_justificar`). El objetivo es refinar el DataFrame `silver` ya sea descartando filas o añadiendo flags.

In [ ]:
for r, (accion, _) in DECISIONES.items():
    # Recupera la condición Spark de la regla y asegura que los NULLs se traten como FALLOS (False).
    # Esto es crucial para que `filter` y `~cond` funcionen correctamente en presencia de valores ausentes.
    cond = F.coalesce(REGLAS[r][1], F.lit(False))
    if accion == "filtrar":
        # Si la decisión es 'filtrar', se eliminan las filas que no cumplen la condición de la regla.
        antes = silver.count()
        silver = silver.filter(cond)
        print(f"filtrar             {r:<30} -{antes - silver.count():>10,} filas")
    else:
        # Si la decisión es 'conservar_con_flag', las filas que no cumplen la regla se mantienen,
        # pero se les añade una nueva columna (un flag binario) que indica su incumplimiento.
        # Esto permite que un modelo aprenda de estas "anomalías" sin descartar los datos.
        silver = silver.withColumn(f"incumple_{r}", (~cond).cast("int"))
        print(f"conservar_con_flag  {r:<30}  columna incumple_{r}")

# Reporte final del estado del DataFrame `silver` después de aplicar todas las decisiones (automáticas y justificadas).
n_silver = silver.count()
print(f"\nSilver: {n_silver:,} filas · {(1 - n_silver/n_bronze)*100:.2f}% descartado desde Bronze")

## 3.6 Gestión de Nulos para `passenger_count`: Marcado de ausencia de valor vs. Imputación.
Aquí se crea una nueva columna `pasajeros_faltante` (un 'flag') para indicar si el `passenger_count` original era nulo. Dado que esta es una operación determinista (se evalúa fila a fila y no 'aprende' nada del conjunto de datos), puede realizarse en esta etapa sin riesgo de data leakage.

In [ ]:
silver = silver.withColumn("pasajeros_faltante", F.col("passenger_count").isNull().cast("int"))
silver.groupBy("pasajeros_faltante").count().show()

# La imputación (por ejemplo, reemplazar nulos con la mediana) es una operación que 'aprende'
# un parámetro de los datos (la mediana). Para evitar la fuga de información (data leakage),
# esta acción se pospone al pipeline de la sección 6, donde se ajustará **exclusivamente**
# con el conjunto de entrenamiento (`train`).

## 3.7 Deduplicación: explorando las tres estrategias principales y sus impactos.
Esta sección compara diferentes métodos para eliminar filas duplicadas, midiendo su rendimiento y analizando las implicaciones de cada uno en términos de determinismo y audibilidad.

In [ ]:
from pyspark.storagelevel import StorageLevel

# Materializa Silver fuera de la RAM cuando sea necesario: las tres estrategias
# reutilizan este mismo conjunto y todas provocan un shuffle.
silver = silver.persist(StorageLevel.MEMORY_AND_DISK)
silver.count()


def cronometrar(etiqueta, fn):
    t0 = time.perf_counter(); n = fn(); print(f"{etiqueta:<34} {n:>12,} filas · {time.perf_counter()-t0:6.2f} s")
    return n

# --- Estrategia 1: Filas idénticas en TODAS las columnas (duplicado "exacto") ---
# `dropDuplicates()` sin argumentos considera que dos filas son duplicadas si todos sus valores
# en todas las columnas coinciden. Esta operación es computacionalmente intensiva (requiere un
# shuffle completo de todo el DataFrame) y, semánticamente, retiene una copia arbitraria de los
# duplicados, lo que la hace no determinista y difícil de auditar.
cronometrar("dropDuplicates() [todas cols]", lambda: silver.dropDuplicates().count())

# --- Estrategia 2: Duplicados por clave de negocio ---
# `dropDuplicates(CLAVE)` define duplicados basándose únicamente en las columnas que forman
# la clave de negocio (`CLAVE`). Si varias filas comparten la misma clave, se conserva una copia
# de forma arbitraria (no determinista). Aunque reduce el alcance del shuffle solo a las columnas
# de la clave, la arbitrariedad en la selección sigue siendo una desventaja en muchos escenarios.
cronometrar("dropDuplicates(CLAVE)", lambda: silver.dropDuplicates(CLAVE).count())

# --- Estrategia 3: Deduplicación determinista usando funciones de ventana (Estrategia Preferida) ---
# Este método utiliza una función de ventana para asignar un número de fila (`row_number`) a cada
# registro dentro de particiones definidas por la clave de negocio (`CLAVE`), y las ordena según
# un criterio específico (aquí, `total_amount` y `fare_amount` descendente). Esto permite
# seleccionar determinísticamente qué copia conservar (ej. la de mayor monto total).
# Es auditable y su comportamiento es predecible, lo que la convierte en la opción ideal para
# la calidad de datos.
w_dup = Window.partitionBy(*CLAVE).orderBy(F.col("total_amount").desc(), F.col("fare_amount").desc())
dedup = silver.withColumn("rn", F.row_number().over(w_dup)).filter("rn = 1").drop("rn")
n_dedup = cronometrar("ventana row_number (elegida)", lambda: dedup.count())

**3.7. Comparación e interpretación de la deduplicación**

| Estrategia | Filas resultantes | Tiempo guardado |
|---|---:|---:|
| `dropDuplicates()` en todas las columnas | 9.294.106 | 45,08 s |
| `dropDuplicates(CLAVE)` | 9.294.088 | 13,67 s |
| Ventana con `row_number` | 9.294.088 | 22,29 s |

**¿Cuál fue más lenta y por qué?** La comparación por todas las columnas fue la más lenta en esta ejecución. Debe distinguir filas completas, lo que puede aumentar el ancho de las claves y el trabajo de agregación. La comparación por clave usa cinco columnas para determinar qué constituye un duplicado. La ventana también agrupa por esa clave y añade un orden para elegir la fila retenida. Estas diferencias explican razonablemente el orden observado, pero una sola medición por método no constituye un benchmark concluyente: influyen caché, plan físico y estado del equipo.

**¿Qué cambia en los datos?** No se eliminan duplicados exactos de Silver, pero sí **18 registros excedentes por clave** con los otros dos métodos. Esto muestra que compartir la clave de negocio no equivale a tener idénticas todas las columnas. Los **70.121 excedentes** del reporte inicial se midieron en Bronze, antes del filtrado; no deben compararse como si las tres estrategias hubieran trabajado sobre ese mismo conjunto.

**¿Por qué se elige la ventana?** Hace explícita la preferencia por `total_amount` descendente y, en segundo lugar, `fare_amount` descendente. Es más auditable que una elección arbitraria por clave, aunque **no garantiza un desempate completo** si ambas columnas coinciden y otras difieren. Además, un monto mayor no demuestra que el registro sea más reciente ni una corrección; es una convención del código, no una secuencia temporal verificada.

En duplicados exactos, escoger cualquier copia produce los mismos valores observables. La arbitrariedad importa especialmente cuando las filas coinciden en la clave pero difieren en el resto. Por último, los tiempos provienen de acciones `count()`: prueban el conteo resultante, no una comparación exhaustiva de todas las filas retenidas.

## 4. Etapa 4: Tres features: temporal, razón y ventana solo-pasado

**Variable objetivo declarada primero.** Sin objetivo no existe la prueba del reloj: la fuga se define respecto a un instante de predicción.

> **Objetivo:** <code>propina_alta</code>: el viaje deja propina mayor al 20% de la tarifa.
> **Instante de predicción:** fin del viaje, antes de que se registre el pago.

Restricción del dominio: la propina en efectivo **no se registra** en TLC (queda en 0). El universo se limita a <code>payment_type = 1</code> (tarjeta); esa restricción es un sesgo de selección y se declara, no se esconde.

In [ ]:
base = (dedup
    .filter(F.col("payment_type") == 1)                      # solo tarjeta: es donde la propina existe
    .withColumn("propina_alta", (F.col("tip_amount") > 0.2 * F.col("fare_amount")).cast("int"))
    .withColumn("fecha", F.to_date("tpep_pickup_datetime")))

base.groupBy("propina_alta").count().show()   # balance de clases: dato para el modelo de la Fase 2


In [ ]:
# F1 · Derivadas del registro (familia 1): la hora del viaje ya está en la fila.
gold = (base
    .withColumn("hora", F.hour("tpep_pickup_datetime"))
    .withColumn("dia_semana", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("es_finde", F.col("dia_semana").isin([1, 7]).cast("int"))
    .withColumn("franja", F.when(F.col("hora") < 6,  "madrugada")
                           .when(F.col("hora") < 12, "mañana")
                           .when(F.col("hora") < 19, "tarde")
                           .otherwise("noche")))

# F2 · Razón entre columnas (familia 1): informa más que cada columna por separado.
gold = gold.withColumn("tarifa_por_milla", F.col("fare_amount") / F.col("trip_distance"))
# NOTA: 'total_amount' NO entra al set de features. total = fare + tip + extras, o sea CONTIENE
# el objetivo. Es el caso de fuga más común y el más difícil de ver: la columna parece inocente.


In [ ]:
# F3 · Agregada por entidad con ventana (familia 3): demanda de la zona en los 7 días ANTERIORES.
diario = (gold.groupBy(F.col("PULocationID").alias("zona"), "fecha")
              .agg(F.count(F.lit(1)).alias("viajes_dia"))
              .withColumn("dia", F.datediff("fecha", F.lit(PERIODO_INI).cast("date"))))

# rangeBetween(-7, -1) sobre días calendario: la cota superior -1 EXCLUYE el día actual.
# Con rowsBetween(-7, -1) se contarían 7 FILAS, no 7 días: si una zona no tuvo viajes un día,
# la ventana se estiraría hacia atrás sin avisar. Con datos reales siempre faltan días.
w7 = Window.partitionBy("zona").orderBy("dia").rangeBetween(-7, -1)
diario = diario.withColumn("demanda_media_7d", F.round(F.avg("viajes_dia").over(w7), 1))

gold = gold.join(diario.selectExpr("zona as PULocationID", "fecha", "demanda_media_7d"),
                 on=["PULocationID", "fecha"], how="left")

gold.select("PULocationID", "fecha", "hora", "franja", "trip_distance", "duracion_min",
            "velocidad_mph", "tarifa_por_milla", "demanda_media_7d", "propina_alta").show(8)


## 5. Etapa 5: <code>EXPLAIN</code> del pipeline: ¿dónde está el shuffle?

Cierre del círculo con el Módulo 1. El pipeline de arriba parece una secuencia de <code>withColumn</code>, pero dos operaciones mueven datos por la red y una tercera pudo evitarlo.

In [ ]:
gold.explain()


### 5b. Lectura del plan físico guardado

Las líneas se cuentan desde `== Physical Plan ==` como **línea 1** en la salida de `gold.explain()`. Las referencias corresponden a esa ejecución; los identificadores internos pueden cambiar al volver a ejecutar.

**1. ¿Cuántos `Exchange` aparecen y qué los provoca?**

Se observan **seis apariciones de `Exchange hashpartitioning`**, todas con **64 particiones**:

| Línea | Clave de partición | Operación que requiere la redistribución |
|---:|---|---|
| 6 | `PULocationID, fecha` | Lado izquierdo del join que añade la demanda histórica. |
| 13 | Las cinco columnas de `CLAVE` | Ventana de deduplicación en la rama izquierda. |
| 53 | `PULocationID, fecha` | Lado derecho del mismo join. |
| 58 | `zona` | Ventana del promedio de los días anteriores por zona. |
| 60 | `PULocationID, fecha` | Agregación que cuenta viajes por zona y fecha. |
| 67 | Las cinco columnas de `CLAVE` | Deduplicación en la rama que construye la tabla diaria. |

Son apariciones en el plan impreso, no un conteo medido de etapas ejecutadas ni del volumen transferido. El plan contiene lecturas de Silver persistido; la deduplicación figura en ambas ramas porque se definió después de esa persistencia.

**2. ¿Cuál corresponde a `demanda_media_7d`?**

El **`Exchange hashpartitioning(zona, 64)` de la línea 58** reúne los registros de cada zona. A continuación aparece el ordenamiento por `zona, dia` en la **línea 57**, y la ventana `RangeFrame, -7, -1` en la **línea 56**. La agrupación coloca cada zona en una partición; el orden temporal permite evaluar los siete días anteriores y excluir el día actual.

El código usa días calendario como límites, pero promedia únicamente los días que tienen una fila en `diario`. No agrega ceros para días sin viajes, por lo que no equivale necesariamente a dividir el total semanal por siete. Además, el agregado se construye desde Gold restringido a tarjeta: describe la actividad observada de esa población, no toda la demanda de taxis.

**3. ¿El join usa broadcast?**

No en el plan guardado: aparece **`SortMergeJoin ... LeftOuter` en la línea 4**, con redistribución de ambos lados en las líneas 6 y 53. El tamaño potencialmente pequeño de la tabla diaria no demuestra que Spark haya elegido broadcast. Para explicar de forma concluyente esa elección haría falta inspeccionar las estimaciones de tamaño y la configuración del optimizador. Un `broadcast()` explícito sería una alternativa de optimización, pero no forma parte del código aquí conservado.

**4. ¿Qué columnas lee Parquet? ¿Se observa column pruning?**

Los `FileScan parquet` de las **líneas 29, 32 y 35** enumeran las **12 columnas seleccionadas en `COLS`**, en lugar de una lectura indiscriminada de todos los campos de origen. Esa proyección es evidencia de poda de columnas. No debe confundirse con filtrado de filas: esos scans muestran `PushedFilters: []`.

El texto incluye otras apariciones de los scans dentro del linaje de las cachés y sus planes iniciales y finales. Su repetición visual **no prueba** que cada archivo se haya vuelto a leer físicamente tantas veces. Las ramas consumidoras muestran `InMemoryTableScan` de Silver en las líneas **16 y 71**.

**5. ¿Qué indica AQE?**

La raíz muestra **`AdaptiveSparkPlan isFinalPlan=false` en la línea 2**: el plan global presentado todavía no es final. Dentro de las relaciones persistidas aparecen también **`isFinalPlan=true` en las líneas 18 y 73**, con sus secciones de plan inicial y final. Esto refleja planes internos ya materializados; no convierte al plan global de Gold en final. Con esta salida no se puede asegurar que AQE haya cambiado el join a broadcast o reducido las 64 particiones de los exchanges externos.

## 6. Fuga de información: la prueba del reloj y el split temporal

Cada feature se somete a la misma pregunta: **en el instante de la predicción, ¿este valor ya existe?**

| Feature | Familia | ¿Existe al final del viaje? | Veredicto |
|---|---|---|---|
| <code>hora</code>, <code>es_finde</code>, <code>franja</code> | Derivada del registro | Sí, desde que empieza | Limpia |
| <code>trip_distance</code>, <code>duracion_min</code>, <code>velocidad_mph</code> | Derivada del registro | Sí, al bajar el pasajero | Limpia |
| <code>tarifa_por_milla</code> | Razón | Sí, el taxímetro cierra antes del pago | Limpia |
| <code>demanda_media_7d</code> | Agregada por entidad | Sí — la ventana corta en <code>-1</code> | Limpia |
| <code>total_amount</code> | Razón | **Incluye la propina** | **Fuga: excluida** |
| <code>tip_amount</code> | — | Es el objetivo | **Objetivo, no feature** |

La segunda forma de fuga no se ve en la tabla de features, sino en el **orden de las operaciones**: un <code>Imputer</code> o un <code>StandardScaler</code> ajustado sobre todo el dataset filtra información del futuro hacia el train. Por eso el split va **antes** del <code>fit</code>, y el split es **temporal**, no aleatorio: un split aleatorio entrena con marzo para predecir enero.

In [ ]:
train = gold.filter(F.col("fecha") <  CORTE_TEST)
test  = gold.filter(F.col("fecha") >= CORTE_TEST)
print(f"train (ene-feb): {train.count():,} · test (mar): {test.count():,}")


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, StandardScaler, VectorAssembler

gold = gold.withColumn("ratecode_str", F.col("RatecodeID").cast("string"))
train = train.withColumn("ratecode_str", F.col("RatecodeID").cast("string"))
test  = test.withColumn("ratecode_str",  F.col("RatecodeID").cast("string"))

A_IMPUTAR   = ["passenger_count", "demanda_media_7d", "tarifa_por_milla", "velocidad_mph"]
IMPUTADAS   = [c + "_imp" for c in A_IMPUTAR]
NUM_DIRECTAS = ["trip_distance", "duracion_min", "hora", "es_finde", "pasajeros_faltante"]
CATEGORICAS = ["franja", "ratecode_str"]

etapas = [
    Imputer(strategy="median", inputCols=A_IMPUTAR, outputCols=IMPUTADAS),
    StringIndexer(inputCols=CATEGORICAS, outputCols=[c + "_idx" for c in CATEGORICAS],
                  handleInvalid="keep"),      # 'keep': categoría no vista en train no rompe el test
    OneHotEncoder(inputCols=[c + "_idx" for c in CATEGORICAS],
                  outputCols=[c + "_ohe" for c in CATEGORICAS], handleInvalid="keep"),
    VectorAssembler(inputCols=IMPUTADAS + NUM_DIRECTAS, outputCol="num_vec"),
    StandardScaler(inputCol="num_vec", outputCol="num_esc", withMean=True, withStd=True),
    VectorAssembler(inputCols=["num_esc"] + [c + "_ohe" for c in CATEGORICAS], outputCol="features"),
]

# fit SOLO con train. Si algún VectorAssembler lanza error por un nulo, es buena noticia:
# significa que una fila esquivó las reglas de la sección 2 y el contrato la detuvo.
prep = Pipeline(stages=etapas).fit(train)


In [ ]:
# Prueba de que los parámetros salieron solo de train: las medianas aprendidas.
print("Medianas del Imputer (ajustadas con ene-feb):")
prep.stages[0].surrogateDF.show()

print("Media y desviación del StandardScaler — 5 primeras:")
print("  media:", [round(float(x), 3) for x in prep.stages[4].mean.toArray()[:5]])
print("  std  :", [round(float(x), 3) for x in prep.stages[4].std.toArray()[:5]])

# El mismo modelo transforma ambos conjuntos. test NUNCA vuelve a ajustar nada.
train_prep, test_prep = prep.transform(train), prep.transform(test)
train_prep.select("features", "propina_alta").show(3, truncate=80)
print("dimensión del vector de features:", len(train_prep.select("features").head()["features"]))


## 7. Registro de variables: el anexo de la Fase 2

Un feature store en su versión mínima: el conjunto de features es un artefacto **con nombre, definición y versión**, no una celda perdida en un notebook. Sin este cuadro, nadie puede reproducir la tabla Gold.

In [ ]:
diccionario = pd.DataFrame([
    ("hora",               "temporal / registro", "Hora de inicio del viaje (0-23)",                        "tpep_pickup_datetime", "—",            "no"),
    ("es_finde",           "temporal / registro", "1 si el viaje inicia sábado o domingo",                  "tpep_pickup_datetime", "—",            "no"),
    ("franja",             "categórica",          "madrugada / mañana / tarde / noche (one-hot)",           "hora",                 "—",            "no"),
    ("ratecode_str",       "categórica",          "Código tarifario TLC (one-hot, handleInvalid=keep)",     "RatecodeID",           "—",            "no"),
    ("trip_distance",      "registro",            "Millas recorridas según taxímetro",                      "trip_distance",        "—",            "no"),
    ("duracion_min",       "razón",               "Minutos entre pickup y dropoff",                         "dropoff - pickup",     "—",            "no"),
    ("velocidad_mph",      "razón",               "trip_distance / (duracion_min / 60)",                    "derivada",             "—",            "no"),
    ("tarifa_por_milla",   "razón",               "fare_amount / trip_distance",                            "derivada",             "—",            "no"),
    ("demanda_media_7d",   "agregada / ventana",  "Viajes diarios promedio de la zona, 7 días previos",     "agregado (zona, fecha)", "[-7d, -1d]",  "no"),
    ("pasajeros_faltante", "flag de calidad",     "1 si passenger_count venía nulo",                        "regla completitud",    "—",            "no"),
    ("passenger_count_imp","registro imputado",   "Pasajeros; mediana aprendida SOLO con train",            "Imputer(median)",      "train",        "no"),
], columns=["feature", "familia", "definición", "origen", "ventana", "riesgo_de_fuga"])

diccionario["version"] = "gold_taxi_v1"
diccionario.to_csv("diccionario_features_v1.csv", index=False)
diccionario


In [ ]:
# Tabla Gold persistida: insumo directo del modelo de la Fase 2 (opcional, ~200 MB).
COLUMNAS_GOLD = (["PULocationID", "DOLocationID", "fecha", "tpep_pickup_datetime"]
                 + NUM_DIRECTAS + A_IMPUTAR + CATEGORICAS + ["propina_alta"])
# gold.select(COLUMNAS_GOLD).write.mode("overwrite").partitionBy("fecha").parquet("gold_taxi_v1")
print("Columnas de la capa Gold:", len(COLUMNAS_GOLD))


## 8. Ejercicios de extensión (Propuestos)

**E1 · Target encoding sin fuga.** Codificar <code>PULocationID</code> (265 categorías: one-hot la infla, StringIndexer le impone un orden falso) por la tasa media de <code>propina_alta</code> de la zona. La media debe calcularse **solo con train** y luego unirse a train y test. Dos preguntas: ¿qué pasa con una zona presente en test y ausente en train, y qué ocurriría si la media se calculara sobre <code>gold</code> completo?

**E2 · Muestreo estratificado.** Rehacer las secciones 4 y 6 sobre una muestra con <code>sampleBy("propina_alta", fractions={0: 0.1, 1: 0.1})</code> y comparar tiempos y la distribución del objetivo. Consigna de la clase: desarrollar con muestra, reportar con el total.

**E3 · <code>rowsBetween</code> vs <code>rangeBetween</code>.** Recalcular <code>demanda_media_7d</code> con <code>rowsBetween(-7, -1)</code> y comparar contra la versión por rango en una zona de baja demanda (p. ej. <code>PULocationID = 5</code>). ¿Cuánto difieren y en qué dirección sesga el error?

**E4 · PCA.** Aplicar <code>PCA(k=10)</code> sobre el one-hot de las 265 zonas y ver cuánta varianza explican los 10 primeros componentes. ¿Vale el costo de perder la interpretabilidad de la zona?

**E5 · Fuga inducida.** Agregar <code>total_amount</code> al <code>VectorAssembler</code>, entrenar una regresión logística y comparar el AUC contra el modelo sin ella. El salto que se observe es la medida exacta de lo que una fuga promete y no cumple en producción.

## Declaración de uso de IA

Se utilizó **Codex** para revisar y mejorar las respuestas redactadas de las secciones **1, 2b, 3.3, 3.4, 3.7 y 5b**.

**Instrucción utilizada:** “Mejorar las respuestas en texto del laboratorio de Osvaldo Neira, manteniendo el código y sus resultados”.

**Validación:** las cifras se contrastaron con las salidas guardadas de perfilado, reporte de calidad, diagnóstico, filtrado y deduplicación. Las referencias de `EXPLAIN` se comprobaron contra los operadores y las líneas del plan físico existente. Se distinguieron resultados observados, interpretaciones e hipótesis, y se compararon las celdas de código y sus salidas con el archivo fuente para verificar su conservación.